# Common Track SMPL Optimized Runtime Demo

This notebook uses the shared `common.smpl` namespace to create an optimized runtime from a tiny `.npz` model, inspect IO cache behavior, and run both Mode 1 and Mode 2 optimization paths.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from common import smpl

OUTPUT_ROOT = REPO_ROOT / "outputs/common/notebooks/smpl_optimized_runtime_demo"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_PATH = OUTPUT_ROOT / "toy_model.npz"

np.savez(
    MODEL_PATH,
    v_template=np.asarray([[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.0, 1.0, 0.0], [0.0, 1.0, 0.0]], dtype=np.float32),
    shapedirs=np.zeros((4, 3, 2), dtype=np.float32),
    posedirs=np.zeros((9, 12), dtype=np.float32),
    J_regressor=np.ones((2, 4), dtype=np.float32) / 4.0,
    weights=np.ones((4, 2), dtype=np.float32) / 2.0,
    parents=np.asarray([-1, 0], dtype=np.int32),
    faces=np.asarray([[0, 1, 2], [0, 2, 3]], dtype=np.int32),
)
MODEL_PATH

## 1. Create An Optimized Runtime And Inspect IO Cache State

In [ ]:
smpl.clear_io_cache()
runtime = smpl.create_runtime(MODEL_PATH, mode="optimized")
diag_after_first = smpl.io_cache_diagnostics()
_ = smpl.create_runtime(MODEL_PATH, mode="optimized")
diag_after_second = smpl.io_cache_diagnostics()
{
    "after_first": vars(diag_after_first),
    "after_second": vars(diag_after_second),
    "runtime_type": type(runtime).__name__,
}

## 2. Run SMPL Mode 1 Through The Common Track

In [ ]:
provision = smpl.initialize_mode1_model(model=runtime, progress=False)
params = dict(provision.params)
params["transl"] = params["transl"].at[0, 0].set(0.5)
run_m1 = smpl.run_mode1_workflow(
    provision,
    output_dir=OUTPUT_ROOT / "mode1",
    prefix="toy",
    params=params,
    steps=3,
    step_size=0.1,
    diagnostics_every=1,
    progress=False,
)
metrics_m1 = json.loads(run_m1.artifacts["metrics"].read_text(encoding="utf-8"))
metrics_m1

## 3. Run SMPL Mode 2 Through The Common Track

In [ ]:
result_m2 = smpl.optimize_mode2(runtime, params, diagnostics_every=1)
{
    "implementation_status": result_m2.implementation_status,
    "n_stages": len(result_m2.phase_summaries),
    "final_objective": float(result_m2.objective_history[-1]),
    "compile_count": runtime.diagnostics().compile_count,
}